In [ ]:
import matplotlib.patches as patches
import pandas as pd
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec # gridspec for nested subfigures
from matplotlib.lines import Line2D
import cartopy.crs as ccrs
from windrose import WindroseAxes
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

from utils.plot_utils import plot_flight_obs, plot_flight_obs_single

level = 'L1'

In [ ]:
# --- Data import

# -- MODIS
modis_dictionary = {'IS22-02':'MOD021KM.A2022081.1125.nc',   # closest full coverage (obs from 11:39:01 to 14:29:56)
                    'IS22-03':'MOD021KM.A2022083.1115.nc',   # closest full coverage (obs from 10:28:59 to 11:10:49)
                    'IS22-04':'MOD021KM.A2022083.1430.nc',   # closest full incloud coverage (obs from 13:09:34 to 14:33:59)
                    'IS22-05':'MOD021KM.A2022085.1100.nc',   # closest full coverage (obs from 09:48:50 to 11:35:35)
                    'IS22-06':'MOD021KM.A2022085.1100.nc',   # closest full coverage (obs from 14:51:19 to 16:53:29)
                    'IS22-07':'MOD021KM.A2022088.1130.nc',   # closest full coverage (obs from 10:02:56 to 11:43:31)
                    'IS22-08':'MOD021KM.A2022089.1035.nc',   # closest full coverage (obs from 15:05:34 to 15:50:54)
                    'IS22-10':'MOD021KM.A2022093.1150.nc',   # closest full coverage (obs from 09:07:47 to 10:56:17)
                    'IS22-11':'MOD021KM.A2022093.1150.nc'    # closest full coverage (obs from 12:38:49 to 14:56:39)
                    } # dictionary over satellite images to use

modis_path = "/home/ninalar/Documents/MC2/procIslasMicrophy/sea_ice_satellite/" # Modis-files folder

# -- Sea ice concentration
sic_path = '/home/ninalar/Documents/MC2/procIslasMicrophy/sea_ice_satellite/asi-n6250-'
sic_file_struct = '-5.4_regridded.nc' # Filename structure

# -- 

# --Paths
save_path = '/home/ninalar/Documents/MC2/procIslasMicrophy-1/save_images/sample_rate/'
processed_path = f'/home/ninalar/Documents/MC2/Results_2022-islas/Processed/ISLAS_processed/{level}' # regular path
main_path = '/home/ninalar/Documents/MC2/Results_2022-islas/Processed/with_distance_from_ice' # path with sea ice distance


In [ ]:
# --- Flight overviews and excemplary MODIS plot
# --- FIGURE 1 in article

# single flight with markings for stratifrom cloud streets and cellcular structure
for islasid in ['IS22-07']:
    #print(islasid)
    ds_incloud_flight = ds_incloud.where(ds_incloud['islasid']==islasid, drop = True)
    ds_all_flight = ds.where(ds['islasid']== islasid, drop = True)
    date = ds_incloud_flight['time'][0].values
    dato = date.astype('datetime64[D]')
    day_of_year = (date.astype('datetime64[D]') - np.datetime64(str(date.astype('datetime64[Y]')))).astype('timedelta64[D]').astype(int) + 1

    # testing for times and day of year for selecting closest MODIS image
    #print(f'Day of year: {day_of_year}')

    min_t = pd.Timestamp(ds_incloud_flight.time.min().values)
    max_t = pd.Timestamp(ds_incloud_flight.time.max().values)
    
    min_time = f'{min_t.hour:02d}:{min_t.minute:02d}'
    max_time = f'{max_t.hour:02d}:{max_t.minute:02d}'
    

    #--- get satellite information for the flight
    filename = modis_dictionary[islasid] # get filename of closest modis from dictionary
    #print(filename)

    ds_modis = xr.open_dataset(modis_path + filename)
    ds_modis_i = ds_modis.isel(time=0) # Choose the first time 

    # get the time information
    modis_t = pd.Timestamp(ds_modis_i.time.values)
    modis_time = f'{modis_t.hour:02d}:{modis_t.minute:02d}'

#--- get sea ice information
    # sea ice for flight
    date = pd.to_datetime(ds_incloud_flight.time[0].values) # get the date and format it correctly
    date = date.strftime('%Y%m%d')

    sic_ds = xr.open_dataset(sic_path  + date + sic_file_struct)
    sic_ds.close()

    # rename data variable and update attributes
    sic_ds['sic'] = sic_ds['__xarray_dataarray_variable__'].assign_attrs(units="Percent", description="Sea Ice Concentration")
    sic_ds = sic_ds.drop_vars(['__xarray_dataarray_variable__'])

    # add some [attributes
    sic_ds.attrs['date'] = date
    sic_ds.attrs['file'] = f'asi-n6250-{date}-5.4_regridded.nc'
    # get sic distances
    sic_select = sic_ds.where(sic_ds.sic>25, drop=True)

    #-- plot together
    fig = plt.figure(figsize=(20, 15), layout="constrained")
    gs = GridSpec(1, 2, figure=fig)
    ax_flights = fig.add_subplot(gs[0,0], projection=ccrs.NorthPolarStereo())
    plot_flight_obs(ds, ds_incloud,sic_max_ds, sic_min_ds,ax=ax_flights, obs=False)
    ax_map = fig.add_subplot(gs[0,1], projection=ccrs.NorthPolarStereo(central_longitude=20))

    # plot satellite image 
    cb = ax_map.pcolormesh(ds_modis_i.lon, ds_modis_i.lat, ds_modis_i.radiance.isel(band=0), transform=ccrs.PlateCarree(), cmap="gray", vmax=150,zorder=0)

    # plot the Flight path and the sea-ice edge border for the given flight

    col_flight, handles, labels = plot_flight_obs_single(full_extent, ds_all_flight, ds_incloud_flight, sic_ds, ax_map)

    # add axes for background color for windrose
    #fax = ax_map.inset_axes([0,0.78,0.37,0.22]) # windrose axes on left top side
    fax = ax_map.inset_axes([0,0.78,0.4,0.22]) # windrose axes on left top side with extention
    #fax = ax_map.inset_axes([0.09,0.78,0.37,0.22]) # windrose axes on left top side, padded to fit (b)
    #fax = ax_map.inset_axes([0.63,0.78,0.37,0.22]) # windrose axes on right top side
    fax.set_facecolor('white')
    fax.set_alpha(0.05)
    fax.set_xticks([])
    fax.set_yticks([])

    # alt fax to add windrose to
    fax_alt = ax_map.inset_axes([0.07,0.78,0.37,0.22]) # windrose axes on left top side, padded to fit (b)
    #fax_alt.set_facecolor('white')
    fax_alt.spines['left'].set_visible(False)
    fax_alt.set_xticks([])
    fax_alt.set_yticks([])

# Plot the windrose
    #wrax = inset_axes(fax, width='70%', height='70%', #original
    wrax = inset_axes(fax_alt, width='70%', height='70%',
                    loc="center",
                    axes_class = WindroseAxes)

    ws = ds_incloud_flight.WS.values
    wd = ds_incloud_flight.WD.values

    wrax.bar(wd, ws, normed=True)
    wrax.set_facecolor('white')
    wrax.set_alpha(0.05)
    wrax.tick_params(labelleft=False)
    wrax.tick_params(axis='x', colors='black')
    for tick in wrax.get_xticklabels():
        tick.set_fontsize(16)
        tick.set_fontweight('bold')

    # add annotations, first create a new axes to put this on
    rect_ax = ax_map.inset_axes([0,0,1,1])
    rect_ax.set_facecolor('None')
    rect_ax.set_xticks([])
    rect_ax.set_yticks([])

    # Cloud streets
    x=0.5
    y=0.55
    rect = patches.Rectangle((x, y), 0.3, 0.1, linewidth=4, edgecolor='white', facecolor='none')
    rect_ax.add_patch(rect)
    rect_ax.text(x+0.29, y+0.1, "Cloud streets", fontsize=20, ha='right', va='bottom',
            bbox=dict(facecolor='white', alpha=1, edgecolor='black', pad=5))

    # add the open cellular
    x=0.3
    y=0.35
    rect = patches.Rectangle((x, y), 0.3, 0.1, linewidth=4, edgecolor='white', facecolor='none')
    rect_ax.add_patch(rect)
    rect_ax.text(x+0.29, y+0.1, "Open Cellular", fontsize=20, ha='right', va='bottom',
            bbox=dict(facecolor='white', alpha=1, edgecolor='black', pad=5))

    # Create custom legend handles for the two titles
    title1 = Line2D([0], [0], color='none', label=f'{islasid} - {dato}\nincloud times: {min_time} - {max_time} \nMODIS image time: {modis_time}', lw=0)  # Empty handle for title

    # Combine the handles and labels
    comb_handles = [title1] + handles
    comb_labels = [title1.get_label()] + labels
    # Add legend to the plot
    l = ax_map.legend(handles=comb_handles, labels=comb_labels, loc='lower left',framealpha=1, fontsize=16)

    #add a) and b) as titles
    ax_flights.set_title('(a)', y=1.0, x=0.04, pad=-30,fontdict={'fontsize': 25})
    fax.set_title('(b)', y=1.0,x=0.09, pad=-30, fontdict={'fontsize': 25})
  
  # Force same height vertical position of plots
    pos_left  = ax_flights.get_position()  # Bbox: x0, y0, x1, y1 in figure coords
    pos_right = ax_map.get_position()

    # Make right axis match left axis height and vertical position
    # [# keep current left x,# match bottom, # keep current width, # match height]
    ax_map.set_position([pos_right.x0, pos_left.y0, pos_right.width,pos_left.height])
  
    fig.savefig(save_path+f'FlightsModisIce_{islasid}_annotated.png', dpi=100, bbox_inches='tight')

In [ ]:
for islasid in np.unique(ds_incloud['islasid'].values):
#for islasid in ['IS22-08']:
    #print(islasid)
    ds_incloud_flight = ds_incloud.where(ds_incloud['islasid']==islasid, drop = True)
    ds_all_flight = ds.where(ds['islasid']== islasid, drop = True)
    date = ds_incloud_flight['time'][0].values
    dato = date.astype('datetime64[D]')
    day_of_year = (date.astype('datetime64[D]') - np.datetime64(str(date.astype('datetime64[Y]')))).astype('timedelta64[D]').astype(int) + 1

    # testing for times and day of year for selecting closest MODIS image
    #print(f'Day of year: {day_of_year}')

    min_t = pd.Timestamp(ds_incloud_flight.time.min().values)
    max_t = pd.Timestamp(ds_incloud_flight.time.max().values)
    
    min_time = f'{min_t.hour:02d}:{min_t.minute:02d}'
    max_time = f'{max_t.hour:02d}:{max_t.minute:02d}'
    

    #--- get satellite information for the flight
    filename = modis_dictionary[islasid] # get filename of closest modis from dictionary
    #print(filename)

    ds_modis = xr.open_dataset(modis_path + filename)
    ds_modis_i = ds_modis.isel(time=0) # Choose the first time 

    # get the time information
    modis_t = pd.Timestamp(ds_modis_i.time.values)
    modis_time = f'{modis_t.hour:02d}:{modis_t.minute:02d}'

    #--- get sea ice information
    # sea ice for flight
    date = pd.to_datetime(ds_incloud_flight.time[0].values) # get the date and format it correctly
    date = date.strftime('%Y%m%d')

    sic_ds = xr.open_dataset(sic_path  + date + sic_file_struct)
    sic_ds.close()

    # rename data variable and update attributes
    sic_ds['sic'] = sic_ds['__xarray_dataarray_variable__'].assign_attrs(units="Percent", description="Sea Ice Concentration")
    sic_ds = sic_ds.drop_vars(['__xarray_dataarray_variable__'])

    # add some [attributes
    sic_ds.attrs['date'] = date
    sic_ds.attrs['file'] = f'asi-n6250-{date}-5.4_regridded.nc'
    # get sic distances
    sic_select = sic_ds.where(sic_ds.sic>25, drop=True)

    #-- plot together
    fig = plt.figure(figsize=(20, 15))
    ax_map = fig.add_subplot(projection=ccrs.NorthPolarStereo(central_longitude=20))

    # plot satellite image 
    cb = ax_map.pcolormesh(ds_modis_i.lon, ds_modis_i.lat, ds_modis_i.radiance.isel(band=0), transform=ccrs.PlateCarree(), cmap="gray", vmax=150,zorder=0)

    # plot the Flight path and the sea-ice edge border for the given flight

    col_flight, handles, labels = plot_flight_obs_single(full_extent, ds_all_flight, ds_incloud_flight, sic_ds, ax_map)

    # add axes for background color for windrose
    fax = ax_map.inset_axes([0,0.78,0.37,0.22])
    fax.set_facecolor('white')
    fax.set_xticks([])
    fax.set_yticks([])

    # Plot the windrose
    wrax = inset_axes(fax, width='70%', height='70%',
                    loc="center",
                    axes_class = WindroseAxes)

    ws = ds_incloud_flight.WS.values
    wd = ds_incloud_flight.WD.values

    wrax.bar(wd, ws, normed=True)
    wrax.set_facecolor('white')
    wrax.tick_params(labelleft=False)
    wrax.tick_params(axis='x', colors='black')
    for tick in wrax.get_xticklabels():
        tick.set_fontsize(16)
        tick.set_fontweight('bold')


    # Create custom legend handles for the two titles
    title1 = Line2D([0], [0], color='none', label=f'{islasid} - {dato}\nincloud times: {min_time} - {max_time} \nMODIS image time: {modis_time}', lw=0)  # Empty handle for title

    # Combine the handles and labels
    comb_handles = [title1] + handles
    comb_labels = [title1.get_label()] + labels
    # Add legend to the plot
    l = ax_map.legend(handles=comb_handles, labels=comb_labels, loc='lower left',framealpha=1, fontsize=16)
  
  
    fig.savefig(save_path+f'FlightModisIce_{islasid}.png', dpi=100, bbox_inches='tight')
